#Data Cleaning and Preprocessing

In [ ]:
# Loading datasets
reports = pd.read_excel("~/talentSentiAnalysis/data/2008-2014_official_reports.xlsx")

positions = pd.read_excel(
    "data/NHL_positions.xlsx",
    sheet_name="Player info+stats",
    usecols=["Draft year", "matched_player_name", "player_position_merged_dataset"]
)

# Standardizing column names
reports.rename(columns={
    "Draft year": "draft_year",
    "Player name": "player_name",
    "Report": "report"
}, inplace=True)

positions.rename(columns={
    "Draft year": "draft_year",
    "matched_player_name": "player_name",
    "player_position_merged_dataset": "position"
}, inplace=True)

In [ ]:
# Where necessary, name entries were manually corrected.
reports["player_name"] = reports["player_name"].str.lower().str.replace(".", "", regex=False)

name_corrections = {
    "corey fienhage": "corey fienhage",
    "eric odell": "eric o'dell"
}

positions['player_name'] = positions['player_name'].replace({"eric odell": "eric o’dell"})
reports['player_name'] = reports['player_name'].replace({"cory fienhage": "corey fienhage"})

In [ ]:
nhl_df = pd.merge(reports, positions, on=["draft_year", "player_name"], how="inner")

# Further examined unmatched columns.
unique_to_reports = reports[~reports['player_name'].isin(positions['player_name'])]
unique_to_pos = positions[~positions['player_name'].isin(reports['player_name'])]

nhl_df['position'] = nhl_df['position'].fillna('Unknown')

# For initial testing purposes, a first batch was taken.
first_batch = nhl_df.groupby("draft_year", group_keys=False).head(2)

# For further analysis, a more complete batch was constructed.
# Two forwards, two defensemen.
second_batch = pd.DataFrame()

for year in first_batch["draft_year"].unique():
    subset_first = first_batch[first_batch["draft_year"] == year]
    subset_pool = nhl_df[nhl_df["draft_year"] == year]

    for pos in ["F", "D"]:
        count_needed = 2 - subset_first[subset_first["position"] == pos].shape[0]
        if count_needed > 0:
            available = subset_pool[
                (~subset_pool["player_name"].isin(subset_first["player_name"])) &
                (subset_pool["position"] == pos)
            ]
            sampled = available.sample(n=count_needed, replace=False)
            second_batch = pd.concat([second_batch, sampled], ignore_index=True)

final_batch = pd.concat([first_batch, second_batch], ignore_index=True)
final_batch.sort_values(by="draft_year", inplace=True)

# A set comprised of two goalies was also constructed.
goalies = nhl_df[nhl_df["position"] == "G"].sample(n=2, replace=False)

In [ ]:
final_batch.to_csv("second_batch.csv", index=False)

final_batch.head()

In [ ]:
goalies.to_csv("goalies.csv", index=False)

goalies.head()

# Model Configuration and Loading

In [ ]:
import os, json, re, time, hashlib, pathlib
import pandas as pd
from google.colab import ai

In [ ]:
PROMPT = """
Your task is to extract concise, self-contained *phrases* that describe hockey player attributes or abilities
from the following scouting paragraph.

Rules:
- Keep phrases intact; do NOT split multi-word phrases (e.g., "quick hands", "board battles").
- Each extracted phrase should describe a single concept, quality, or descriptor of the player
  (e.g., skill, role, physical trait), typically 2–6 words long.
  Avoid long phrases, full sentences, or multiple ideas.
- Prefer noun or adjective phrases (e.g., "good vision", "strong on puck") rather than full clauses.
- Normalize capitalization and plurals (e.g., "Quick Hands" → "quick hands").
- Always include physical or role descriptors (e.g., "pint-sized defender", "big-bodied winger", "offensive defenseman")
  when they describe the player's identity or physical makeup, even if neutral.
- Assign a sentiment label for each phrase: one of ["strength","weakness","neutral"].
  • strength = clearly positive contribution/quality
    (e.g., "good vision", "strong shot")
  • weakness = clearly negative limitation/shortcoming
    (e.g., "poor balance", "slow feet")
  • neutral  = factual/role-fit/ambiguous mentions
    (e.g., "plays center position")
- Assign exactly one trait for each phrase from:
  ["psychological","physical","tactical","technical","miscellaneous"].
  • psychological = mindset, competitiveness, poise, confidence, consistency, under-pressure decisions
  • physical      = size, speed, strength, agility, endurance, balance
  • tactical      = positioning, anticipation/reads, forecheck/backcheck, board battles, PK/PP roles, routes
  • technical     = puck skills (hands/handling), shot, passing, edges/skating mechanics, faceoffs, net-front tips
  • miscellaneous = anything not covered above (style notes, role fit, context, general statements)
- If a phrase fits multiple traits, choose the one that best reflects the primary aspect being evaluated.
- If polarity within a phrase is mixed, label it "neutral" unless one side clearly dominates.
- Ensure phrases are lowercase, trimmed of punctuation, and unique within the output list.
- Do not include explanations, reasoning, or summaries.
- Output *only* valid JSON with this exact schema:

{
  "items": [
    {"phrase": "<string>", "label": "<strength|weakness|neutral>", "trait": "<psychological|physical|tactical|technical|miscellaneous>"},
    ...
  ]
}

Text:
```{TEXT}```
""".strip()


In [ ]:
CACHE_DIR = pathlib.Path("colab_ai_cache")
CACHE_DIR.mkdir(exist_ok=True)

def _first_json_block(s: str) -> str:
    m = re.search(r"\{.*\}", s, flags=re.DOTALL)
    return m.group(0) if m else s

def _norm_label(x: str) -> str:
    x = (x or "").strip().lower()
    return x if x in {"strength","weakness","neutral"} else "neutral"

def _norm_trait(x: str) -> str:
    x = (x or "").strip().lower()

    allowed = {"psychological","physical","tactical","technical","miscellaneous"}
    return x if x in allowed else "miscellaneous"

def _cache_key(text: str, model_name: str) -> str:
    h = hashlib.md5((model_name + "|" + text).encode()).hexdigest()
    return f"{h}.json"

In [ ]:
def extract_items_for_paragraph(text: str,
                                model_name: str = "google/gemini-2.0-flash",
                                sleep_sec: float = 0.2,
                                use_cache: bool = True) -> list:
    """Call Gemini once for a paragraph; return normalized list of dicts."""
    text = (text or "").strip()
    if not text:
        return []

    key = _cache_key(text, model_name)
    cache_path = CACHE_DIR / key
    if use_cache and cache_path.exists():
        try:
            with open(cache_path, "r") as f:
                cached = json.load(f)
            return cached  # already normalized
        except Exception:
            pass  # fall through and recompute

    prompt = PROMPT.replace("{TEXT}", text)
    raw = ai.generate_text(prompt, model_name=model_name)
    raw_str = str(raw)
    json_text = _first_json_block(raw_str)

    try:
        data = json.loads(json_text)
    except Exception:
        # fallback: try to coerce with common fixes
        # (You can extend with more robust cleaning if needed)
        raise ValueError("Model did not return valid JSON. Raw:\n" + raw_str[:1000])

    items = data.get("items", [])
    cleaned = []
    for it in items:
        phrase = (it.get("phrase") or "").strip()
        if not phrase:
            continue
        cleaned.append({
            "phrase": phrase,
            "label": _norm_label(it.get("label")),
            "trait": _norm_trait(it.get("trait")),
        })

    # cache normalized result
    if use_cache:
        with open(cache_path, "w") as f:
            json.dump(cleaned, f, ensure_ascii=False)

    # tiny pause to be gentle with quotas
    if sleep_sec:
        time.sleep(sleep_sec)

    return cleaned

In [ ]:
MODEL = "google/gemini-2.0-flash"

#LLM-Based Phrase Extraction, Trait Classification, and Sentiment Labelling

In [ ]:
# We set sample_df to either the final batch, or to goalies.

sample_df = pd.read_csv("final_batch.csv")

# sample_df = pd.read_csv("goalies.csv")

sample_df.head()


,draft_year,player_name,report,position
0,2008,aj jenks,"The hard working, big-bodied Jenks has shown s...",F
1,2008,aaron ness,The pint sized defender is pure offense from t...,D
2,2008,yann sauve,The big mobile defender has a great overall pa...,D
3,2008,greg nemisz,Nemisz is a big bodied player who can use his ...,F
4,2009,alex chiasson,"Immense size…the puck protection skills, great...",F


In [ ]:
sample_df["items"] = sample_df["report"].apply(lambda txt: extract_items_for_paragraph(txt, model_name=MODEL))

In [ ]:
meta_cols = [c for c in sample_df.columns if c not in {"items"}]
rows = []
for _, row in sample_df.iterrows():
    base = {k: row[k] for k in meta_cols}
    for it in (row["items"] or []):
        rows.append({**base, **it})

phr_df = pd.DataFrame(rows) if rows else pd.DataFrame(columns=meta_cols + ["phrase","label","trait"])
display(phr_df.head())

,draft_year,player_name,report,position,phrase,label,trait
0,2008,aj jenks,"The hard working, big-bodied Jenks has shown s...",F,hard working,strength,psychological
1,2008,aj jenks,"The hard working, big-bodied Jenks has shown s...",F,big-bodied,neutral,physical
2,2008,aj jenks,"The hard working, big-bodied Jenks has shown s...",F,constant threat in both zones,strength,tactical
3,2008,aj jenks,"The hard working, big-bodied Jenks has shown s...",F,physical bite,strength,physical
4,2008,aj jenks,"The hard working, big-bodied Jenks has shown s...",F,delivering a big hit,strength,physical


In [ ]:
if not phr_df.empty:
    print("Counts by label:")
    display(phr_df["label"].value_counts().rename_axis("label").reset_index(name="count"))

    print("Counts by trait:")
    display(phr_df["trait"].value_counts().rename_axis("trait").reset_index(name="count"))

    if {"label","trait"}.issubset(phr_df.columns):
        print("Label × Trait matrix:")
        display(pd.crosstab(phr_df["label"], phr_df["trait"]))

    # Example: per-player/per-year rollups if those columns exist
    group_keys = [c for c in ["year","player"] if c in phr_df.columns]
    if group_keys:
        print("Per-entity counts (label):")
        display(phr_df.groupby(group_keys + ["label"]).size().reset_index(name="count").head(20))

        print("Per-entity counts (trait):")
        display(phr_df.groupby(group_keys + ["trait"]).size().reset_index(name="count").head(20))

Counts by label:


,label,count
0,strength,429
1,neutral,134
2,weakness,125


Counts by trait:


,trait,count
0,physical,237
1,tactical,156
2,technical,141
3,psychological,102
4,miscellaneous,52


Label × Trait matrix:


trait,miscellaneous,physical,psychological,tactical,technical
label,,,,,
neutral,27,56,5,36,10
strength,22,134,66,100,107
weakness,3,47,31,20,24


In [ ]:
phr_df.to_csv("all_phrases_exploded.csv", index=False, encoding="utf-8-sig")